<a href="https://colab.research.google.com/github/anupz-0/Faster_Whisper_ASR_livestreaming/blob/main/FasterOP_Whisper_ASR_livestreaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [ ]:
# ============================================
# CELL 1: Install Dependencies
# ============================================
!pip install -q fastapi uvicorn[standard] websockets python-multipart
!pip install -q faster-whisper ctranslate2 silero-vad
!pip install -q symspellpy python-Levenshtein pyngrok nest-asyncio
print("✅ All dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 135.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.0 MB/s eta 0:00:00
✅ All dependencies installed!


ADD *server_streaming_optimized.pyd text* and *dictionaries* Files

In [ ]:
# ============================================
# CELL 2: Configure for GPU
# ============================================
import torch

with open('server_streaming_optimized.py', 'r') as f:
    content = f.read()

# Enable GPU
content = content.replace('DEVICE = "cpu"', 'DEVICE = "cuda" if torch.cuda.is_available() else "cpu"')
content = content.replace('COMPUTE_TYPE = "int8"', 'COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"')
content = content.replace('WHISPER_MODEL = "medium"', 'WHISPER_MODEL = "large-v3"')

with open('server_streaming_optimized.py', 'w') as f:
    f.write(content)

print(f"✅ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


✅ GPU Available: True
   GPU Model: Tesla T4
   GPU Memory: 15.6 GB


Use your ngrok auth token from *(https://dashboard.ngrok.com/get-started/your-authtoken)*

In [ ]:
# ============================================
# CELL 3: Setup ngrok
# ============================================
from pyngrok import ngrok

# ⚠️ GET YOUR TOKEN FROM: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "36RbeatuwZOmL9PHFEk27HFhkPH_7zNmwLLJtETYGdv34Xzhf"  # ← REPLACE THIS!

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok authenticated!")

✅ ngrok authenticated!


In [ ]:
# ============================================
# CELL 4: Start Server
# ============================================
import nest_asyncio
import subprocess
import threading
import time

nest_asyncio.apply()

def run_server():
    subprocess.run(['python', 'server_streaming_optimized.py'])

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("⏳ Starting server (loading model...)")
time.sleep(20)
print("✅ Server is running!")

⏳ Starting server (loading model...)
✅ Server is running!


In [ ]:
# ============================================
# CELL 5: Get Public URL
# ============================================
public_url = ngrok.connect(8000, bind_tls=True)
ws_url = str(public_url).replace('https://', 'wss://')

print("\n" + "="*70)
print("🎉 YOUR ASR SERVER IS LIVE!")
print("="*70)
print(f"\n📍 WebSocket URL (copy this):")
print(f"   {ws_url}/ws/asr")
print(f"\n📍 HTTP URL:")
print(f"   {public_url}")
print("\n" + "="*70)
print("\n📋 NEXT STEPS:")
print("   1. Copy the WebSocket URL above")
print("   2. Open index_optimized.html")
print("   3. Replace WS_URL with your WebSocket URL")
print("   4. Open the HTML file in browser")
print("   5. Start speaking!")
print("\n⏱️  Session expires in ~12 hours")
print("="*70 + "\n")


🎉 YOUR ASR SERVER IS LIVE!

📍 WebSocket URL (copy this):
   NgrokTunnel: "wss://vixenish-vihaan-unstrategically.ngrok-free.dev" -> "http://localhost:8000"/ws/asr

📍 HTTP URL:
   NgrokTunnel: "https://vixenish-vihaan-unstrategically.ngrok-free.dev" -> "http://localhost:8000"


📋 NEXT STEPS:
   1. Copy the WebSocket URL above
   2. Open index_optimized.html
   3. Replace WS_URL with your WebSocket URL
   4. Open the HTML file in browser
   5. Start speaking!

⏱️  Session expires in ~12 hours



In [ ]:

# ============================================
# CELL 6: Keep Server Running
# ============================================
import time

print("🔄 Server is running...")
print("💡 Keep this cell running to keep server alive")
print("⏹️  Click 'Stop' button to shutdown\n")

counter = 0
try:
    while True:
        time.sleep(60)
        counter += 1
        print(f"⏰ Running for {counter} minutes...", end="\r")
except KeyboardInterrupt:
    print("\n✅ Server stopped!")

🔄 Server is running...
💡 Keep this cell running to keep server alive
⏹️  Click 'Stop' button to shutdown

